# The Attention Layer That Learned to Rotate
**Companion to the Medium post "Two Mirrors Make a Rotation" (2026-09-22)**

Linear attention replaces attention's growing KV cache with a fixed-size
"whiteboard" matrix S updated every step:

$$S_t = (I - \beta_t k_t k_t^\top)\,\mathrm{Diag}(\alpha_t)\,S_{t-1} + \beta_t k_t v_t^\top$$

Moonshot AI's **Kimi Delta Attention (KDA)** made the decay gate
$\alpha_t$ *per-channel* instead of one scalar — and the Sept 22, 2026
**Complex KDA** paper (arXiv 2609.24797) showed that allowing signed gates
$\alpha \in [-1, 1]$ with $\beta \in [0, 2]$ lets a *single* update
perform a 2D rotation. This notebook verifies that claim with pure NumPy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# The paper's Figure 1 construction: beta=2 (Householder mirror),
# k=(1,1)/sqrt(2), channel-wise signed gate Diag(-1, 1)
beta = 2.0
k = np.array([1., 1.]) / np.sqrt(2)
alpha = np.array([-1., 1.])

H = np.eye(2) - beta * np.outer(k, k)   # Householder reflection
D = np.diag(alpha)                      # coordinate reflection
A = H @ D                               # one Complex KDA transition

print("Transition matrix A =\n", np.round(A, 4))
print("Eigenvalues:", np.round(np.linalg.eigvals(A), 4))
print("-> complex-conjugate pair on the unit circle: a true rotation")
print("Rotation angle:", round(np.degrees(np.angle(np.linalg.eigvals(A)[0])), 1), "degrees")

p = np.array([0.3, 1.0])
Ap = A @ p
R90 = np.array([[0., -1.], [1., 0.]])
print("A·p == +90° rotation of p:", np.allclose(Ap, R90 @ p))
print("Length preserved (non-expansive):", np.isclose(np.linalg.norm(Ap), np.linalg.norm(p)))

## 1. Why the *channel-wise* gate is the whole trick
Gated DeltaNet uses one scalar gate α for every channel. A scalar commutes with
everything, so its transition matrix is symmetric — and symmetric matrices have
**real** eigenvalues only. No complex eigenvalues means no rotations, ever.
KDA's per-channel gate breaks that symmetry, and signed entries (α = −1) supply
the second mirror. Let's watch the spectrum change:

In [ ]:
def spectrum(alpha_vec, beta=2.0, k=k):
    H = np.eye(2) - beta * np.outer(k, k)
    return np.linalg.eigvals(H @ np.diag(alpha_vec))

configs = {
    "Gated DeltaNet  (scalar gate, α=0.9)": np.array([0.9, 0.9]),
    "KDA             (positive gate)    ": np.array([0.9, 0.5]),
    "Complex KDA     (signed gate)      ": np.array([-1., 1.]),
}
fig, ax = plt.subplots(figsize=(6, 6))
theta = np.linspace(0, 2*np.pi, 200)
ax.plot(np.cos(theta), np.sin(theta), "k--", alpha=0.3, label="unit circle")
for name, a in configs.items():
    ev = spectrum(a)
    ax.scatter(ev.real, ev.imag, s=90, label=f"{name}  λ={np.round(ev,2)}")
ax.axhline(0, color="#ccc", lw=0.8); ax.axvline(0, color="#ccc", lw=0.8)
ax.set_aspect("equal"); ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
ax.set_xlabel("real"); ax.set_ylabel("imaginary")
ax.set_title("One transition's eigenvalues: only the signed gate leaves the real line",
             loc="left", weight="bold")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()
print("Only Complex KDA's eigenvalues are complex (±i) — the fingerprint of rotation.")

## 2. State tracking: turn the rotation into a counter
The paper's expressivity ladder: 1D reflections → parity; **2D rotations →
modular addition**. If the transition really rotates, iterating it should count:
12 quarter-turns = back to start. We simulate a tiny 2-channel memory cell that
counts modulo 12, using *nothing* but repeated Complex KDA updates.

In [ ]:
steps = 12
p0 = np.array([1.0, 0.0])   # "count = 0" lives on the +x axis
traj = [p0]
v = p0.copy()
for t in range(steps):
    v = A @ v               # one Complex KDA update per token
    traj.append(v.copy())
traj = np.array(traj)

angles = np.degrees(np.arctan2(traj[:, 1], traj[:, 0])) % 360
counts = np.round(angles / 90).astype(int) % 4 * 3 + np.round(angles / 360).astype(int) * 0
# simpler: count quarter-turns directly
turns = np.round(np.unwrap(np.arctan2(traj[:,1], traj[:,0])) / (np.pi/2)).astype(int) % 12
print("token:   ", list(range(steps + 1)))
print("counter: ", list(turns))
print("Returned to start after 12 tokens:", np.allclose(traj[-1], traj[0]))

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(traj[:, 0], traj[:, 1], "o-", color="#1a73e8", lw=1.5)
for i, pt in enumerate(traj[:-1]):
    ax.annotate(str(i), tuple(pt), fontsize=9, weight="bold",
                xytext=(6, 6), textcoords="offset points")
ax.set_aspect("equal"); ax.axhline(0, color="#ccc", lw=0.8); ax.axvline(0, color="#ccc", lw=0.8)
ax.set_title("12 Complex KDA updates trace a 12-hour clock face — a mod-12 counter", loc="left", weight="bold")
plt.tight_layout(); plt.show()

## 3. Why cheaper alternatives can't do this
**DeltaProduct₂** (Siems et al., 2025) achieves the same 2D rotation, but it
composes *two* delta-rule updates per token — roughly twice the recurrent work.
Complex KDA folds the second mirror into the already-free channel gate, so the
update stays **diagonal-plus-rank-one** and keeps 96–97% of standard KDA's
kernel throughput (paper, §1). Same expressivity, one update instead of two:

In [ ]:
# cost model: one diagonal-plus-rank-one update ~ O(n*d) per token; DeltaProduct2 doubles the rank-one part
labels = ["Complex KDA\n(1 update, rank-1)", "DeltaProduct2\n(2 updates, rank-2)"]
work = [1.0, 2.0]
fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(labels, work, color=["#1a73e8", "#9334e6"], alpha=0.9)
ax.set_ylabel("recurrent work per token (relative)")
ax.set_title("Same rotation expressivity, half the per-token recurrent work", loc="left", weight="bold")
for b, v in zip(bars, work):
    ax.text(b.get_x() + b.get_width()/2, v + 0.05, f"{v:.0f}×", ha="center", weight="bold")
ax.set_ylim(0, 2.5)
plt.tight_layout(); plt.show()
print("And per the paper, the range extension changes KDA kernels by a few lines,")
print("retaining 96-97% of standard KDA throughput — the rotation is nearly free.")

## 4. The system-level payoff: Kimi Linear
The transition above is the core of **Kimi Linear** (Moonshot AI, arXiv
2510.26692): a 3B-activated / 48B-total hybrid of KDA and MLA layers — the
first linear attention to beat full attention under an identical training
recipe, with up to 75% less KV cache and up to 6× decoding throughput at a
1M-token context. Same-recipe numbers from the tech report:

In [ ]:
labels = ["MMLU-Pro", "RULER 128k"]
kimi, mla, gdn = [84.3, 51.0], [81.3, 47.2], [80.5, 47.9]
x = np.arange(len(labels)); w = 0.24
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w, gdn, w, color="#9aa0a6", label="Gated DeltaNet-H")
ax.bar(x, mla, w, color="#e8710a", label="Full attention (MLA)")
ax.bar(x + w, kimi, w, color="#1a73e8", label="Kimi Linear (KDA + MLA)")
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel("score")
ax.set_title("Kimi Linear vs full attention, same training recipe", loc="left", weight="bold")
ax.legend(fontsize=9); ax.set_ylim(0, 100)
plt.tight_layout(); plt.show()

## Takeaway
The delta rule erases with a mirror; a per-channel signed gate mirrors again —
and mirror × mirror is a rotation. That single geometric fact is what lifts
Complex KDA from "parity only" to tracking S₃, S₄, A₅ and modular addition with
one layer, matching DeltaProduct₂ at half the recurrent cost. When the attention
layer can rotate, it can count — and counting is the seed of every longer
chain of stateful reasoning.

*Sources: "Complex KDA" (Siems et al., arXiv 2609.24797, Sept 22, 2026);
"Kimi Linear" technical report (Kimi Team / Moonshot AI, arXiv 2510.26692).
Benchmark figures are from the Kimi Linear tech report under an identical
training recipe.*